In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import kagglehub

print('TensorFlow version:', tf.__version__)

path = kagglehub.dataset_download('zalando-research/fashionmnist')
print('Path to dataset files:', path)


In [ ]:
class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Training dtype:', X_train.dtype)
print('Pixel range:', X_train.min(), 'to', X_train.max())


In [ ]:
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

X_train_cnn = X_train_norm[..., np.newaxis]
X_test_cnn = X_test_norm[..., np.newaxis]

print('Normalized training shape:', X_train_cnn.shape)
print('Normalized test shape:', X_test_cnn.shape)
print('Normalized range:', X_train_cnn.min(), 'to', X_train_cnn.max())


In [ ]:
plt.figure(figsize=(8, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(class_names[y_train[i]])
    plt.axis('off')
plt.tight_layout()
plt.show()


## MLP Baseline

The baseline flattens each 28×28 image into 784 values, followed by Dense(128) and a 10-class softmax output. It is trained for 5 epochs as specified.


In [ ]:
mlp = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

mlp.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

mlp.summary()
mlp_params = mlp.count_params()
print('MLP parameters:', mlp_params)


In [ ]:
mlp_history = mlp.fit(
    X_train_cnn, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.20,
    verbose=1
)

mlp_test_loss, mlp_test_acc = mlp.evaluate(X_test_cnn, y_test, verbose=0)
print(f'MLP Test Loss: {mlp_test_loss:.4f}')
print(f'MLP Test Accuracy: {mlp_test_acc:.4f}')


##  LeNet-5 Inspired CNN


In [ ]:
cnn = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(6, (5, 5), padding='same', activation='relu'),
    layers.AveragePooling2D(pool_size=(2, 2), strides=2),
    layers.Conv2D(16, (5, 5), padding='valid', activation='relu'),
    layers.AveragePooling2D(pool_size=(2, 2), strides=2),
    layers.Flatten(),
    layers.Dense(120, activation='relu'),
    layers.Dense(84, activation='relu'),
    layers.Dense(10, activation='softmax')
])

cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn.summary()
cnn_params = cnn.count_params()
print('CNN parameters:', cnn_params)


In [ ]:
cnn_history = cnn.fit(
    X_train_cnn, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.20,
    verbose=1
)

cnn_test_loss, cnn_test_acc = cnn.evaluate(X_test_cnn, y_test, verbose=0)
print(f'CNN Test Loss: {cnn_test_loss:.4f}')
print(f'CNN Test Accuracy: {cnn_test_acc:.4f}')


##  CNN Training Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(cnn_history.history['loss'], label='Training Loss')
axes[0].plot(cnn_history.history['val_loss'], label='Validation Loss')
axes[0].set_title('CNN Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(cnn_history.history['accuracy'], label='Training Accuracy')
axes[1].plot(cnn_history.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_title('CNN Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


##  MLP vs CNN Comparison


In [ ]:
comparison = pd.DataFrame({
    'Model': ['Flattened MLP', 'LeNet-5 CNN'],
    'Parameters': [mlp_params, cnn_params],
    'Test Loss': [mlp_test_loss, cnn_test_loss],
    'Test Accuracy': [mlp_test_acc, cnn_test_acc]
})

comparison


##  Classification Report


In [ ]:
cnn_predictions = cnn.predict(X_test_cnn, verbose=0)
y_pred = np.argmax(cnn_predictions, axis=1)

print(classification_report(
    y_test,
    y_pred,
    target_names=class_names,
    digits=4
))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
plt.title('LeNet-5 CNN Confusion Matrix')
plt.tight_layout()
plt.show()


##  Conclusion

The MLP treats the image as a flat vector and therefore does not explicitly preserve local spatial relationships. The LeNet-5 inspired CNN applies convolution and pooling before classification, allowing it to learn spatially local visual features. The comparison table above provides the measured parameter count, test loss, and test accuracy for both models.
